# Missing Value and Outlier Analysis

**Objective:** Analyze missing values, fix missing data, and detect outliers in a dataset.

**Dataset:** Synthetic retail/customer dataset

This notebook is Colab-ready and saves tables, metrics, and visual outputs under
`results/`. Public datasets or compact sample datasets are used so the workflow
remains reproducible.


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
rng = np.random.default_rng(42)


In [ ]:
df = pd.DataFrame(
    {
        "age": rng.normal(35, 10, 250),
        "income": rng.normal(55000, 14000, 250),
        "spend_score": rng.normal(60, 18, 250),
        "visits": rng.poisson(6, 250),
        "segment": rng.choice(["A", "B", "C"], 250),
    }
)
df.loc[rng.choice(df.index, 25, replace=False), "income"] = np.nan
df.loc[rng.choice(df.index, 18, replace=False), "spend_score"] = np.nan
df.loc[rng.choice(df.index, 8, replace=False), "segment"] = np.nan
df.loc[rng.choice(df.index, 5, replace=False), "income"] *= 3

missing_summary = df.isna().mean().mul(100).reset_index()
missing_summary.columns = ["feature", "missing_percent"]
display(missing_summary)


In [ ]:
numeric_cols = df.select_dtypes(include="number").columns
cat_cols = df.select_dtypes(exclude="number").columns

df_fixed = df.copy()
df_fixed[numeric_cols] = SimpleImputer(strategy="median").fit_transform(df_fixed[numeric_cols])
df_fixed[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(df_fixed[cat_cols])

outlier_rows = []
for col in numeric_cols:
    q1, q3 = df_fixed[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((df_fixed[col] < lower) | (df_fixed[col] > upper)).sum()
    outlier_rows.append({"feature": col, "outlier_count": int(count), "lower": lower, "upper": upper})

outliers = pd.DataFrame(outlier_rows)
missing_summary.to_csv(RESULTS_DIR / "missing_summary.csv", index=False)
outliers.to_csv(RESULTS_DIR / "outlier_summary.csv", index=False)
df_fixed.to_csv(RESULTS_DIR / "cleaned_dataset.csv", index=False)
display(outliers)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(df.isna(), cbar=False, ax=axes[0])
axes[0].set_title("Missing Value Map")
sns.boxplot(data=df_fixed[numeric_cols], ax=axes[1])
axes[1].set_title("Outlier Check After Imputation")
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "missing_outlier_dashboard.png", dpi=180)
plt.show()
